# 06b · DOSE at ADM1: step B first stages

Fork `fork/corridors-dose`, Part 2. Rules: [`docs/successors/dose-prespec.md`](../../docs/successors/dose-prespec.md)
§3, §3a, §3b; outcomes recorded in §4. **Loads no temperature**: the region panel
(`src/analysis/dose.py`) reads only lights and row counts from the cell panel.

◇ Step B gate, per instrument: first stage of ln(1 + mean lights) on the instrument, with
precipitation `P_a`, region + country × year FE, weighted by the region's panel cells,
clustered by region. Continue to LST with that instrument only if **F ≥ 10**.

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT))
warnings.filterwarnings("ignore")
from src.analysis import dose as D

df = D.build_region_panel()
s = D.step_sample(df)
print(f"step sample: {len(s):,} region-years, {s.GID_1.nunique():,} regions, "
      f"{s.GID_0.nunique()} countries, {s.year.min()}-{s.year.max()}")

## Exposure (prespec §3a, information, no gate)

Regions whose instrument changes over the years, within D1's coverage sample (2002–2020) and
within the step sample.

In [ ]:
d1 = df[df.d1_covered == 1]
pd.DataFrame([
    dict(instrument=k, column=c,
         d1_regions=D.exposed_regions(d1, c)[0], d1_countries=D.exposed_regions(d1, c)[1],
         sample_regions=D.exposed_regions(s, c)[0], sample_countries=D.exposed_regions(s, c)[1])
    for k, c in D.INSTRUMENTS.items()])

## ◇ Gate: the three first stages

In [ ]:
gate = pd.DataFrame([dict(name=k, **D.first_stage(s, "log_ntl_r", c)) for k, c in D.INSTRUMENTS.items()])
gate["passes"] = gate.F >= 10
gate.round(5)

## Information (not gated)

In [ ]:
info = [
    ("mine count, ln(1 + count)", D.first_stage(s, "log_ntl_r", "log1p_mine_count")),
    ("mine count, country clusters", D.first_stage(s, "log_ntl_r", "mine_count", vcov=D.CL_CTY)),
    ("mine x price, moving exposure", D.first_stage(s, "log_ntl_r", "price_moving")),
    ("mine x price, country clusters", D.first_stage(s, "log_ntl_r", "price_frozen", vcov=D.CL_CTY)),
    ("favoritism, any day", D.first_stage(s, "log_ntl_r", "fav_any")),
    ("favoritism, lagged one year", D.first_stage(s, "log_ntl_r", "fav_share_lag1")),
    ("favoritism, country clusters", D.first_stage(s, "log_ntl_r", "fav_share", vcov=D.CL_CTY)),
]
pd.DataFrame([dict(check=k, **r) for k, r in info]).drop(columns=["endog"]).round(5)

Mine × price, leaving one commodity out (only 12 price series drive the instrument).

In [ ]:
import pyfixest  # noqa: F401  (D.first_stage imports it; fail early if missing)
exp = pd.read_parquet(f"{D.INSTR}/adm1_price_exposure.parquet")
rows = []
for c in sorted(exp.commodity.unique()):
    t = s.assign(price_loo=D.price_leave_one_out(s, c))
    rows.append(dict(left_out=c, **D.first_stage(t, "log_ntl_r", "price_loo")))
pd.DataFrame(rows).drop(columns=["endog", "instrument"]).round(5)